# 3. Let small patches share context
**Predict → run → inspect → explain.** The code is provided. Read the question before pressing Run all; edit only the small controls when the guide asks.

Predict: how many 4×4 patches fit in a 28×28 image? What does position add?

Use Python 3.11+ in Jupyter or a standard Colab Python runtime. CPU is the default. This source has been statically checked, not executed by the authoring assistant; no accuracy or runtime is promised.

## 1. Your controls
The fixed small subsets make the same images available to all three classifiers. Keep the test set out of architecture selection.

In [ ]:
ARCH = 'transformer'
EPOCHS = 3  # @param {type:"integer"}
BATCH_SIZE = 64
MODEL_SEED = 17
SPLIT_SEED = 17  # Keep this fixed across the digit notebooks.
TRAIN_PER_CLASS = 500
VAL_PER_CLASS = 100
TEST_PER_CLASS = 100
USE_GPU = False  # CPU first; accelerator access is optional.


## 2. Prepare the tools
Expand this cell if you want to see the setup. Missing packages are installed into the current notebook environment; restart the kernel if the installer requests it.

In [ ]:
import os, sys, subprocess, importlib.util, json, hashlib, platform
from pathlib import Path
from uuid import uuid4
if not USE_GPU:
    os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
packages = {'tensorflow': 'tensorflow>=2.16,<3', 'numpy': 'numpy>=1.26,<3',
            'matplotlib': 'matplotlib>=3.7,<4', 'pandas': 'pandas>=2,<3',
            'sklearn': 'scikit-learn>=1.4,<2'}
missing = [p for module, p in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])
assert 1 <= EPOCHS <= 20 and BATCH_SIZE >= 1
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import accuracy_score, confusion_matrix
if tuple(int(v) for v in tf.__version__.split('.')[:2]) < (2, 16):
    raise RuntimeError('Use TensorFlow 2.16+; install requirements-notebooks.txt and restart the kernel.')
keras.utils.set_random_seed(MODEL_SEED)
tf.config.experimental.enable_op_determinism()
OUT = Path('outputs')/'ch17'/ARCH/str(uuid4())
OUT.mkdir(parents=True, exist_ok=False)
config = dict(architecture=ARCH, model_seed=MODEL_SEED, split_seed=SPLIT_SEED,
              epochs=EPOCHS, batch_size=BATCH_SIZE, use_gpu=USE_GPU)
software = dict(python=platform.python_version(), tensorflow=tf.__version__,
                numpy=np.__version__, pandas=pd.__version__)
print('Evidence directory:', OUT)
print('Devices:', tf.config.list_physical_devices())
print('Versions:', software)


## 3. Meet real public data
[MNIST via Keras](https://keras.io/api/datasets/mnist/) supplies official training and test images. The notebook selects 500 training, 100 validation, and 100 test images per digit. Training/validation IDs refer to the official training array; test IDs refer to the separate official test array. Pixel division by 255 is fixed conversion.

The loader does not provide writer IDs. This is image classification with the canonical source split, **not participant-validated neuroscience**. MNIST attribution: Yann LeCun and Corinna Cortes; the Keras page states CC BY-SA 3.0. Do not confuse that dataset license with the software's license.

In [ ]:
(images, labels), (official_test_images, official_test_labels) = keras.datasets.mnist.load_data()
rng = np.random.default_rng(SPLIT_SEED)
train_ids, val_ids, test_ids = [], [], []
for digit in range(10):
    ids = rng.permutation(np.flatnonzero(labels == digit))
    assert TRAIN_PER_CLASS + VAL_PER_CLASS <= len(ids)
    train_ids.extend(ids[:TRAIN_PER_CLASS])
    val_ids.extend(ids[TRAIN_PER_CLASS:TRAIN_PER_CLASS+VAL_PER_CLASS])
    candidates = rng.permutation(np.flatnonzero(official_test_labels == digit))
    assert TEST_PER_CLASS <= len(candidates)
    test_ids.extend(candidates[:TEST_PER_CLASS])
train_ids, val_ids, test_ids = map(np.array, (train_ids, val_ids, test_ids))
assert set(train_ids).isdisjoint(val_ids)
# /255 is fixed unit conversion, not a statistic fitted on all images.
X_train = images[train_ids, ..., None].astype('float32') / 255
X_val = images[val_ids, ..., None].astype('float32') / 255
X_test = official_test_images[test_ids, ..., None].astype('float32') / 255
y_train, y_val, y_test = labels[train_ids], labels[val_ids], official_test_labels[test_ids]
split = dict(dataset='MNIST via Keras', training_source='official train',
             test_source='official test', train_image_ids=train_ids.tolist(),
             validation_image_ids=val_ids.tolist(), test_image_ids=test_ids.tolist(),
             writer_ids_available=False,
             selected_train_sha256=hashlib.sha256(images[train_ids].tobytes()+labels[train_ids].tobytes()).hexdigest(),
             selected_validation_sha256=hashlib.sha256(images[val_ids].tobytes()+labels[val_ids].tobytes()).hexdigest(),
             selected_test_sha256=hashlib.sha256(official_test_images[test_ids].tobytes()+official_test_labels[test_ids].tobytes()).hexdigest())
(OUT/'split_manifest.json').write_text(json.dumps(split, indent=2))
config.update(train_per_class=TRAIN_PER_CLASS, validation_per_class=VAL_PER_CLASS,
              test_per_class=TEST_PER_CLASS)
fig, axes = plt.subplots(2, 5, figsize=(9, 4))
for digit, ax in enumerate(axes.ravel()):
    i = np.flatnonzero(y_train == digit)[0]
    ax.imshow(X_train[i, ..., 0], cmap='gray'); ax.set_title(f'Target {digit}'); ax.axis('off')
plt.tight_layout(); plt.show()
print('Train / validation / test:', X_train.shape, X_val.shape, X_test.shape)


## 4. The model is a readable recipe
Explain the input shape, each representation change, and the ten-logit output before running.

In [ ]:
class FixedPosition(layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        position = np.arange(49)[:, None]
        rates = 10000.0 ** (-2 * (np.arange(32)//2) / 32)
        angle = position * rates[None, :]
        values = np.where(np.arange(32)[None, :] % 2 == 0, np.sin(angle), np.cos(angle))
        self.position = tf.constant(values[None, ...], dtype=tf.float32)
    def call(self, inputs):
        return inputs + self.position

inputs = keras.Input(shape=(28, 28, 1))
# Nonoverlapping 4x4 patches -> 7x7 grid -> 49 tokens, each with 32 features.
x = layers.Conv2D(32, 4, strides=4)(inputs)
x = layers.Reshape((49, 32))(x)
x = FixedPosition()(x)
h = layers.LayerNormalization()(x)
h = layers.MultiHeadAttention(num_heads=2, key_dim=16)(h, h)
x = layers.Add()([x, h])
h = layers.LayerNormalization()(x)
h = layers.Dense(64, activation='relu')(h)
h = layers.Dense(32)(h)
x = layers.Add()([x, h])
x = layers.GlobalAveragePooling1D()(x)
model = keras.Model(inputs, layers.Dense(10)(x))
model.summary()


## 5. Learn and inspect validation
Training changes weights. Validation loss chooses the checkpoint and controls early stopping. Test data are not used by `fit`.

In [ ]:
model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3),
              loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])
history = model.fit(X_train, y_train, validation_data=(X_val, y_val),
                    epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=2,
                    callbacks=[keras.callbacks.EarlyStopping(
                        monitor='val_loss', patience=2, restore_best_weights=True)])
selected_epoch = int(np.argmin(history.history['val_loss'])) + 1
pd.DataFrame(history.history).to_csv(OUT/'history.csv', index=False)
plt.plot(history.history['loss'], label='training')
plt.plot(history.history['val_loss'], label='validation')
plt.xlabel('Training epoch (zero-based plot index)'); plt.ylabel('Cross-entropy')
plt.legend(); plt.tight_layout(); plt.savefig(OUT/'learning_curves.png', dpi=160); plt.show()
print('Selected epoch:', selected_epoch, '| Executed epochs:', len(history.history['loss']))


## 6. Evaluate the locked teaching experiment
Run this after fixing the model. Scores from multiple exploratory notebook runs do not form a fresh independent test of a winner chosen after viewing them.

In [ ]:
probabilities = tf.nn.softmax(model.predict(X_test, verbose=0), axis=-1).numpy()
predictions = probabilities.argmax(axis=1)
majority = int(np.bincount(y_train, minlength=10).argmax())
accuracy = float(accuracy_score(y_test, predictions))
baseline_accuracy = float(np.mean(y_test == majority))
print({'test_image_accuracy': accuracy, 'training_majority_baseline': baseline_accuracy})
print('Confusion matrix: rows = observed digit, columns = predicted digit')
print(confusion_matrix(y_test, predictions, labels=np.arange(10)))
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
# Fixed display examples; no selection of an especially successful gallery.
for ax, i in zip(axes.ravel(), range(0, len(X_test), max(1, len(X_test)//10))):
    ax.imshow(X_test[i, ..., 0], cmap='gray')
    ax.set_title(f'True {y_test[i]} / predicted {predictions[i]}'); ax.axis('off')
plt.tight_layout(); plt.savefig(OUT/'prediction_gallery.png', dpi=160); plt.show()
pd.DataFrame({'official_test_image_id': test_ids, 'target': y_test,
              'prediction': predictions, 'max_probability': probabilities.max(axis=1)}).to_csv(
                  OUT/'predictions.csv', index=False)
np.save(OUT/'probabilities.npy', probabilities)
report = dict(task='MNIST digit classification', test_image_accuracy=accuracy,
              majority_baseline_accuracy=baseline_accuracy, selected_epoch=selected_epoch,
              executed_epochs=len(history.history['loss']), parameters=int(model.count_params()),
              evaluation_unit='image; no supplied writer IDs', clinical_validation=False)


### Inspect patching before interpreting attention
The grid is a tokenization choice. Attention between patches is computational weighting, not a brain-connectivity measurement.

In [ ]:
plt.imshow(X_val[0, ..., 0], cmap='gray')
for edge in np.arange(3.5, 28, 4):
    plt.axvline(edge, color='#167D91'); plt.axhline(edge, color='#167D91')
plt.title('7 × 7 patches = 49 tokens'); plt.show()


## 7. Preserve the evidence
The selected weights, split IDs, predictions, curves, configuration and versions travel together. Weights alone cannot resume the original training trajectory.

In [ ]:
model.save_weights(OUT/'selected.weights.h5')
(OUT/'architecture.json').write_text(model.to_json())
(OUT/'config.json').write_text(json.dumps(config, indent=2))
(OUT/'model_card.md').write_text(
    '# Teaching model\n\nPublic data, educational use. No clinical validation. '
    'Validation loss selected the checkpoint; test data did not. '
    'Weights alone are not a complete training-resume checkpoint. '
    'Architecture is supplied in this notebook; retain the notebook and environment.\n')
report.update(software=software, config=config, review_status='review_required', accepted=False)
report['artifact_sha256'] = {p.name: hashlib.sha256(p.read_bytes()).hexdigest()
                            for p in sorted(OUT.iterdir()) if p.is_file()}
(OUT/'report.json').write_text(json.dumps(report, indent=2))
print('Saved:', sorted(p.name for p in OUT.iterdir()))
print('Download this directory before ending an ephemeral Colab session; keep it outside Git.')


## Explain before moving on
1. Trace 49 tokens × 32 features and two attention heads. Explain why key_dim=16 means 16 dimensions per head, not 16 heads.
2. What did the training data teach, and what did validation choose?
3. Why is an image-level test different from holding out EEG participants?

Keep your answer and saved evidence. The full assignment and next notebook are linked from the chapter README. A low or surprising score is a prompt to inspect the experiment, not to hide it.